# Enriched feature signature
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rotskoff-group/idiom/blob/v1/cookbook/notebooks/enriched_feature_signature.ipynb)

Compare positive IDRs with a length-matched background and export enriched feature IDs for the SAE reward notebook.


## Setup
A GPU runtime is recommended: **Runtime → Change runtime type → GPU**. CPU execution is supported but slower.

Run cells in order. Installation is self-contained; no repository clone or account is needed. If Colab requests a session restart after installation, restart before running the imports. The first model load downloads weights.


In [ ]:
import sys

!"{sys.executable}" -m pip install -q "idiom[cookbook] @ git+https://github.com/rotskoff-group/idiom.git@v1"

In [1]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

from idiom import IDiomSAE
from idiom.utils.notebook_helpers import example_file, idr_sequence, save_run, write_fasta

started = time.perf_counter()

## Settings
Choose positive and background sets relevant to your question. The example compares activation and repression domains. Annotated protein headers use `_IDR_start-end` with 1-based, inclusive coordinates.


In [2]:
POSITIVE_FASTA = None # Default: activation-domain example FASTA
BACKGROUND_FASTA = None # Default: repression-domain example FASTA
POSITIVE_MODE = "idr" # "idr" for isolated IDRs; "annotated" for proteins with IDR spans
BACKGROUND_MODE = "idr" # Input format for the background FASTA
SAE_ID = "jxliu2/idiomsae-300M-L18-k32" # Pretrained SAE and its frozen host model
DEVICE = "auto" # "cpu" or "cuda"; "auto" uses an available GPU
MAX_POSITIVE = 32 # Maximum sampled positive IDRs
MAX_BACKGROUND = 64 # Maximum length-matched background IDRs
BATCH_SIZE = 2 # Sequences per SAE encoding batch
SEED = 0 # Random seed
TOP_N = 30 # Maximum features to include after filtering
NAME = "ad_vs_rd" # Signature name to use as SIGNATURE_NAME in the SAE reward notebook
CASE = "top30" # Signature group to use as CASE in the SAE reward notebook
OUT_DIR = Path("enrichment_outputs") / time.strftime("%Y%m%d-%H%M%S") # New timestamped folder per run

## Prepare inputs
Exact duplicate IDRs are collapsed, and positive IDRs are excluded from the background. Length matching does not control homology or composition. Full input audits are saved as CSV files.


In [3]:
from idiom.sae.features import (
    FeatureDataset,
    enrich,
    feature_counts,
    save_enrichment,
    select_features,
    write_signature,
)
from idiom.sae.features.enrichment import length_match
from idiom.utils.notebook_helpers import load_feature_inputs

if OUT_DIR.exists() and any(OUT_DIR.iterdir()):
    raise ValueError("Use a new or empty output directory.")
OUT_DIR.mkdir(parents=True, exist_ok=True)
positive_path = POSITIVE_FASTA or example_file("effector/ad.fasta", Path("example_inputs"))
background_path = BACKGROUND_FASTA or example_file("effector/rd.fasta", Path("example_inputs"))
sae = IDiomSAE.from_pretrained(SAE_ID, device=DEVICE)
if sae.fim_mode != "unprompted" or sae.region != "idr":
    raise ValueError("Use an unprompted IDR SAE for this workflow.")

positive_pool, positive_audit = load_feature_inputs(
    positive_path, POSITIVE_MODE, max_length=sae.model.cfg.max_seq_len - 4
)
background_pool, background_audit = load_feature_inputs(
    background_path, BACKGROUND_MODE, max_length=sae.model.cfg.max_seq_len - 4
)
positive_idrs = {idr_sequence(r) for r in positive_pool}
overlap = [r.accession for r in background_pool if idr_sequence(r) in positive_idrs]
background_audit.loc[background_audit.record_id.isin(overlap), "status"] = "overlaps positives"
background_pool = [r for r in background_pool if idr_sequence(r) not in positive_idrs]
if MAX_POSITIVE < 1 or MAX_BACKGROUND < 1:
    raise ValueError("Sample limits must be positive.")
chosen = np.random.default_rng(SEED).permutation(len(positive_pool))[:MAX_POSITIVE]
positives = [positive_pool[i] for i in sorted(chosen)]
positive_audit.to_csv(OUT_DIR / "positive_input_audit.csv", index=False)
background_audit.to_csv(OUT_DIR / "background_input_audit.csv", index=False)
if not positives or not background_pool:
    raise ValueError("Need nonempty positive and nonoverlapping background sets; review the input audits.")
background = length_match(positives, background_pool, n=MAX_BACKGROUND, rng=np.random.default_rng(SEED))
for label, records, audit in (
    ("positive", positives, positive_audit),
    ("background", background, background_audit),
):
    audit["selected"] = audit.record_id.isin([r.accession for r in records])
    audit.to_csv(OUT_DIR / f"{label}_input_audit.csv", index=False)
    write_fasta(records, OUT_DIR / f"{label}_selected.fasta")
    print(f"{label}: {len(records)} selected IDRs")

positive: 32 selected IDRs
background: 64 selected IDRs


## Encode and compare
A feature counts once per sequence if it fires anywhere. Selection uses log₂ odds ratios, approximate p-values with Benjamini–Hochberg correction, and the default boundary-artifact filter. A small example may select no features.


In [4]:
pos_fd = FeatureDataset(sae.build_feature_dataset(positives, OUT_DIR / "fd_positive", batch_size=BATCH_SIZE))
bg_fd = FeatureDataset(
    sae.build_feature_dataset(background, OUT_DIR / "fd_background", batch_size=BATCH_SIZE)
)
a, n_pos = feature_counts(pos_fd)
b, n_neg = feature_counts(bg_fd)
result = enrich(a, n_pos, b, n_neg, sae.sae.num_latents)
selection = select_features(result, n=TOP_N, feature_dir=bg_fd)
ids = selection["ids"]
_ = save_enrichment(OUT_DIR / "enrichment.npz", result, selection)

## Inspect and export


In [5]:
table = pd.DataFrame({key: value for key, value in result.items() if isinstance(value, np.ndarray)})
table.insert(0, "feature_id", range(len(table)))
table["selected"] = selection["selected"]
table.to_csv(OUT_DIR / "enrichment.tsv", sep="	", index=False)
columns = ["feature_id", "log2or", "fdr", "prev_pos", "prev_neg", "selected"]
preview = table.loc[table.selected if ids else table.active, columns]
display(preview.sort_values("log2or", ascending=False).head(3))
if ids:
    write_signature(
        OUT_DIR / "signature.json",
        {NAME: ids},
        case=CASE,
        provenance=dict(sae=SAE_ID, seed=SEED, n_pos=n_pos, n_background=n_neg),
    )
    print(f"Signature {NAME!r}: {len(ids)} features (case {CASE!r}); first IDs: {ids[:5]}")
else:
    print("No features passed the filters; no signature was written.")

,feature_id,log2or,fdr,prev_pos,prev_neg,selected
11248,11248,4.665674,0.000012,0.59375,0.046875,True
9172,9172,4.292384,0.000630,0.43750,0.031250,True
853,853,4.269461,0.000012,0.68750,0.093750,True


Signature 'ad_vs_rd': 9 features (case 'top30'); first IDs: [11248, 9172, 853, 12523, 2883]


## Results
`enrichment.tsv` and `enrichment.npz` contain the comparison and selection results. `signature.json` is written when features pass the filters. Use that file in [SAE reward design](https://colab.research.google.com/github/rotskoff-group/idiom/blob/v1/cookbook/notebooks/rl_with_sae_rewards.ipynb), with the same `SAE_ID`, `CASE`, and signature name (`NAME` here becomes `SIGNATURE_NAME` there).

`run.json` records the settings and package versions. Open the output folder in Colab’s **Files** pane to download results. Download them before the runtime ends, or copy them to mounted Drive. Saved notebook previews do not include the exported files.


Saved previews show a CPU example run. Run the cells to create the exported files.

In [6]:
save_run(
    OUT_DIR,
    dict(
        sae=SAE_ID,
        device=DEVICE,
        seed=SEED,
        positive=str(positive_path),
        background=str(background_path),
        positive_mode=POSITIVE_MODE,
        background_mode=BACKGROUND_MODE,
        n_pos=n_pos,
        n_background=n_neg,
        top_n=TOP_N,
        name=NAME,
        case=CASE,
    ),
    elapsed=time.perf_counter() - started,
)
print("Results folder:", OUT_DIR)

Results folder: enrichment_outputs/20260921-134208
